# KaraokeSep na nuvem

Separa as faixas na placa de vídeo do Google, de graça, e devolve um zip pronto
para descompactar na pasta `saida` do seu KaraokeSep.

---

## Só isto: dois cliques

**1.** Menu `Ambiente de execução` → `Alterar o tipo de ambiente de execução` →
acelerador **T4 GPU** → Salvar.

**2.** Escolha o modo abaixo e clique no ▶ da célula.

Quando aparecer o botão **Escolher arquivos**, selecione as músicas. O resto
acontece sozinho: instala, separa, empacota e baixa.

---

| | No seu notebook | Aqui |
|---|---|---|
| 4 faixas | htdemucs | BS-Roformer-SW |
| Tempo por música | ~5 min | menos de 1 min |
| Modo Profundo | mais de 2 h | ~3 min |

A placa não muda a qualidade, muda a velocidade — e com ela os modelos pesados
ficam viáveis, o que na prática melhora o resultado.

In [ ]:
#@markdown ### Configuração
MODO = "padrao"  #@param ["karaoke", "padrao", "padrao_6", "profundo"]
FORMATO = "FLAC"  #@param ["FLAC", "MP3", "WAV"]
REFAZER = False  #@param {type:"boolean"}
SEPARAR_VOZES = False  #@param {type:"boolean"}
#@markdown **karaoke** 2 faixas · **padrao** 4 faixas · **padrao_6** 6 faixas ·
#@markdown **profundo** 10 faixas
#@markdown
#@markdown FLAC guarda sem perda. MP3 320 baixa em um terço do tamanho.
#@markdown
#@markdown `REFAZER` separa de novo o que já foi separado nesta sessão.
#@markdown
#@markdown `SEPARAR_VOZES` divide a voz em masculina e feminina, para dueto.
#@markdown Funciona quando cada um canta um trecho; falha quando cantam juntos
#@markdown ou quando os timbres são parecidos. Custa uma passada a mais.

import os, subprocess, sys, time
from pathlib import Path

PASSO = 0
def etapa(texto):
    global PASSO
    PASSO += 1
    print(f"\n{'='*62}\n  {PASSO}. {texto}\n{'='*62}")

# ------------------------------------------------------------------ #
etapa("Conferindo a placa de video")
# ------------------------------------------------------------------ #
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    print("   NENHUMA PLACA ATIVA.\n")
    print("   Menu: Ambiente de execucao > Alterar o tipo de ambiente")
    print("   Escolha T4 GPU, salve, e rode esta celula de novo.")
    raise SystemExit("sem placa")
print("   ", r.stdout.strip())

# ------------------------------------------------------------------ #
etapa("Instalando")
# ------------------------------------------------------------------ #
try:
    import audio_separator, audioread           # noqa: F401
    print("   Ja instalado nesta sessao.")
except ImportError:
    print("   Sao mais de 2 GB. De 2 a 4 minutos. Nao feche a aba.")
    t0 = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "audio-separator[gpu]", "audioread"], check=True)
    print(f"   Pronto em {time.time()-t0:.0f} s.")

import torch
print(f"   PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("CUDA indisponivel; confira o acelerador")

# ------------------------------------------------------------------ #
etapa("Enviando as musicas")
# ------------------------------------------------------------------ #
EXT = {".mp3",".flac",".wav",".m4a",".ogg",".opus",".aac",".wma",".aiff"}
ENTRADA = Path("/content/entrada"); ENTRADA.mkdir(exist_ok=True)

def listar():
    return sorted(f for f in ENTRADA.iterdir()
                  if f.is_file() and f.suffix.lower() in EXT)

if listar():
    print(f"   {len(listar())} musica(s) ja enviadas nesta sessao; reaproveitando.")
    print("   Para trocar, apague a pasta /content/entrada e rode de novo.")
else:
    print("   Clique em ESCOLHER ARQUIVOS e selecione as musicas.")
    print("   Segure Ctrl para marcar varias. Nao feche a aba durante o envio.")
    import shutil as _sh
    from google.colab import files
    for nome in files.upload():
        _sh.move(nome, ENTRADA/nome)

musicas = listar()
if not musicas:
    raise SystemExit("nenhuma musica enviada")
for f in musicas:
    print(f"   {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

# ------------------------------------------------------------------ #
etapa(f"Separando em modo {MODO}")
# ------------------------------------------------------------------ #
import functools, re, shutil
from audio_separator.separator import Separator

SAIDA = Path("/content/saida"); SAIDA.mkdir(exist_ok=True)
MODELOS = Path("/content/modelos"); MODELOS.mkdir(exist_ok=True)

CANDIDATOS = {
    "voz":     ["model_bs_roformer_ep_317_sdr_12.9755.ckpt",
                "melband_roformer_inst_v1e_plus.ckpt",
                "MDX23C-8KFFT-InstVoc_HQ.ckpt",
                "UVR-MDX-NET-Inst_HQ_4.onnx"],
    "stems":   ["BS-Roformer-SW.ckpt", "htdemucs_6s.yaml", "htdemucs_ft.yaml"],
    "drumsep": ["MDX23C-DrumSep-aufr33-jarredou.ckpt"],
    "backing": ["mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
                "bs_roformer_karaoke_anvuew.ckpt"],
    # voz masculina x feminina. SDR 7.29, bem abaixo dos stems instrumentais:
    # espere vazamento audivel onde os dois cantam ao mesmo tempo.
    "vozes":   ["bs_roformer_male_female_by_aufr33_sdr_7.2889.ckpt",
                "male_female"],
}
MAPA = {"vocals":"vocal","vocal":"vocal","lead vocals":"vocal_principal",
        "instrumental":"instrumental","no vocals":"instrumental",
        "drums":"bateria","bass":"baixo","guitar":"guitarra","piano":"piano",
        "other":"outros","back vocals":"coro","backing vocals":"coro",
        "kick":"bumbo","snare":"caixa","toms":"toms","cymbals":"pratos",
        "hh":"chimbal","hihat":"chimbal","ride":"conducao","crash":"crash",
        "male":"voz_masculina","female":"voz_feminina"}

sep = Separator(log_level=40, model_file_dir=str(MODELOS), output_format=FORMATO)

def normalizar(s): return re.sub(r"[-_.\s]+", "", s.lower())

@functools.lru_cache(maxsize=1)
def catalogo():
    achados = set()
    def andar(no):
        if isinstance(no, str):
            b = no.lower()
            if (b.endswith((".ckpt",".onnx",".pth",".yaml")) and "://" not in no
                    and not b.startswith("config_") and "_config." not in b):
                achados.add(no)
        elif isinstance(no, dict):
            for k, v in no.items(): andar(k); andar(v)
        elif isinstance(no, (list, tuple, set)):
            for v in no: andar(v)
    andar(sep.list_supported_model_files())
    return sorted(achados)

def resolver(papel):
    disp = {normalizar(m): m for m in catalogo()}
    for cand in CANDIDATOS[papel]:
        alvo = normalizar(cand)
        if alvo in disp: return disp[alvo]
        for chave, orig in disp.items():
            if alvo in chave: return orig
    return None

def classificar(caminho):
    m = re.findall(r"\(([^)]+)\)", Path(caminho).stem)
    bruto = re.sub(r"[_-]+", " ", (m[-1] if m else "").strip().lower()).strip()
    if bruto in MAPA: return MAPA[bruto]
    for k, v in MAPA.items():
        if k in bruto: return v
    return re.sub(r"[^a-z0-9]+", "_", bruto) or "faixa"

def passada(entrada, papel, destino, criados, mapa=None):
    modelo = resolver(papel)
    if not modelo: raise RuntimeError(f"sem modelo para o papel {papel}")
    print(f"      modelo: {modelo}")
    bruto = destino/"_bruto"; bruto.mkdir(parents=True, exist_ok=True)
    antes = set(bruto.glob("*"))
    sep.output_dir = str(bruto)
    sep.load_model(model_filename=modelo)
    sep.separate(str(entrada))
    resultado = {}
    for origem in [p for p in bruto.glob("*") if p not in antes and p.is_file()]:
        nome = classificar(origem)
        if mapa: nome = mapa.get(nome, nome)
        alvo = destino/f"{nome}{origem.suffix}"
        # Uma passada da cascata nao pode apagar o que a anterior criou; mas
        # isso so vale para ESTA execucao, senao reprocessar duplica tudo.
        if alvo.resolve() == Path(entrada).resolve() or alvo.resolve() in criados:
            raiz = Path(entrada).stem.lower()
            nome = f"{raiz}_{nome}" if not nome.startswith(raiz) else f"{nome}_2"
            alvo = destino/f"{nome}{origem.suffix}"
            print(f"      conflito de nome, gravando como {alvo.name}")
        if alvo.exists(): alvo.unlink()
        shutil.move(str(origem), str(alvo))
        criados.add(alvo.resolve())
        resultado[nome] = alvo
    shutil.rmtree(bruto, ignore_errors=True)
    return resultado

def limpar_nome(t):
    t = re.sub(r"(?i)\s*(?:tuberipper|y2mate|ytmp3|savefrom|mp3juices|onlymp3)[a-z0-9._-]*", " ", t)
    t = re.sub(r"(?i)\s*\.(?:net|com|com\.br|org|io|cc|to|me)\b", " ", t)
    t = re.sub(r"(?i)\s*[\(\[][^\)\]]*(?:official|video|audio|lyric|hd|4k)[^\)\]]*[\)\]]", " ", t)
    t = re.sub(r"^\s*\d{1,3}\s*[-._)]\s*", "", t)
    t = re.sub(r'[<>:"/\\|?*]', "_", t)
    return re.sub(r"\s{2,}", " ", t).strip(" -._") or "musica"

feitas, puladas, falharam = 0, 0, []
for i, origem in enumerate(musicas, 1):
    nome = limpar_nome(origem.stem)
    pasta = SAIDA/nome
    ja = pasta.is_dir() and any(f.suffix.lower() in EXT for f in pasta.iterdir())
    if ja and not REFAZER:
        print(f"\n[{i}/{len(musicas)}] {nome}  —  ja separada, pulando")
        puladas += 1
        continue

    shutil.rmtree(pasta, ignore_errors=True)
    pasta.mkdir(parents=True, exist_ok=True)
    print(f"\n[{i}/{len(musicas)}] {nome}")
    t0 = time.time(); criados = set()
    try:
        if MODO == "karaoke":
            res = passada(origem, "voz", pasta, criados)
        else:
            res = passada(origem, "stems", pasta, criados)
            if MODO == "profundo":
                if "bateria" in res:
                    print("   dividindo a bateria...")
                    res.update(passada(res["bateria"], "drumsep", pasta, criados))
                if "vocal" in res:
                    print("   separando voz principal e coro...")
                    res.update(passada(res["vocal"], "backing", pasta, criados,
                               mapa={"vocal":"vocal_principal","instrumental":"coro"}))
        if SEPARAR_VOZES:
            alvo = res.get("vocal_principal") or res.get("vocal")
            if alvo:
                print("   dividindo a voz em masculina e feminina...")
                res.update(passada(alvo, "vozes", pasta, criados))
            else:
                print("   sem faixa de voz; separacao de vozes pulada")
        print(f"   {time.time()-t0:.0f} s  ->  {len(res)} faixas: "
              f"{', '.join(sorted(res))}")
        feitas += 1
    except Exception as e:
        import traceback; traceback.print_exc()
        falharam.append(nome)

# ------------------------------------------------------------------ #
etapa("Conferindo o resultado")
# ------------------------------------------------------------------ #
total = 0.0
for pasta in sorted(SAIDA.iterdir()):
    if not pasta.is_dir(): continue
    arq = sorted(f for f in pasta.iterdir() if f.is_file())
    mb = sum(f.stat().st_size for f in arq)/1e6
    total += mb
    print(f"   [{pasta.name}]  {len(arq)} faixas, {mb:.0f} MB")
    print(f"      {', '.join(f.stem for f in arq)}")
print(f"\n   {feitas} separadas, {puladas} puladas"
      + (f", {len(falharam)} com erro: {', '.join(falharam)}" if falharam else ""))
print(f"   Total em disco: {total:.0f} MB")

# ------------------------------------------------------------------ #
etapa("Baixando")
# ------------------------------------------------------------------ #
if total == 0:
    print("   Nada para baixar.")
else:
    if total > 400:
        print("   AVISO: acima de 400 MB o download costuma falhar.")
        print("   Se cair, rode de novo com FORMATO = MP3.")
    shutil.make_archive("/content/stems", "zip", str(SAIDA))
    tam = Path("/content/stems.zip").stat().st_size/1e6
    print(f"   stems.zip ({tam:.0f} MB) — o download comeca sozinho.")
    from google.colab import files as _f
    _f.download("/content/stems.zip")

print(f"\n{'='*62}")
print("  Pronto. Descompacte o stems.zip dentro da pasta 'saida'")
print("  do KaraokeSep e abra o festa.bat.")
print(f"{'='*62}")

---

## Para separar mais músicas

Apague a pasta das músicas antigas e rode a célula de novo:

```python
import shutil
shutil.rmtree("/content/entrada", ignore_errors=True)
```

Ou marque `REFAZER` se quiser reprocessar as mesmas em outro modo.

O que já foi separado é pulado automaticamente, então rodar de novo depois de
uma queda continua de onde parou.

## Sobre a separação de vozes

Só faz sentido em dueto com timbres diferentes e trechos alternados. Aí o
modelo quase só precisa decidir quem está cantando em cada momento.

Onde não funciona: harmonia em terças, refrão com os dois no uníssono, duas
vozes do mesmo registro. O SDR de 7,29 já avisa — o modelo de baixo, para
comparação, chega a 14,6.

## Se der problema

**Parou dizendo que não há placa** — ligue a T4 em `Ambiente de execução` →
`Alterar o tipo de ambiente de execução`.

**O Colab desconectou** — a sessão grátis cai por inatividade e tem teto de
horas por dia. Rode a célula de novo; a instalação e as músicas enviadas se
perdem, mas o fluxo é o mesmo.

**Upload muito lento** — é a sua internet subindo, uns 30 MB por música em
FLAC. Se estiver inviável, mande MP3 320, aceitando a perda na fonte.

**Download falhou** — rode com `FORMATO = MP3`, ou separe menos músicas por vez.

**Linhas vermelhas sobre `libcublas` ou `libcudart`** — pode ignorar. É o
`onnxruntime` reclamando de bibliotecas que os modelos usados aqui não precisam.

**Faltou alguma faixa no modo festa** — rode o `diagnostico.bat` no seu PC.